# 🚜 Module B: Crop Yield Prediction using Regression
*Smart Crop & Agriculture Analyzer*


In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP & CORE LIBRARY IMPORTS
# =============================================================================
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Preprocessing & Pipelines
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Scikit-Learn Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, mean_absolute_error, mean_squared_error,
    r2_score, silhouette_score
)

# Plotting & Display configurations
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("✅ All core Machine Learning and Data Science libraries imported successfully!")


## 1. Load Indian Agricultural Dataset & Clean Outliers


In [ ]:
# 2.2 Inspect Crop Recommendation Data Quality
print("=== Crop Recommendation Dataset (First 5 Rows) ===")
display(crop_df.head())

print("\nMissing Values Check:")
print(crop_df.isnull().sum())

print(f"\nDuplicate Records: {crop_df.duplicated().sum()}")
print(f"Total Unique Crop Categories: {crop_df['label'].nunique()} crops")
print(f"Crop Categories: {sorted(crop_df['label'].unique().tolist())}")

print("\nClass Distribution (Balanced: 100 samples per crop):")
print(crop_df['label'].value_counts().head(5))


In [ ]:
# 3.1 Cleaning the Crop Yield Dataset
df_yield = yield_raw_df.dropna(subset=["Production", "Area"]).copy()
df_yield = df_yield[df_yield["Area"] > 0].copy()

# Strip whitespace from categorical text fields
for col in ["State_Name", "District_Name", "Season", "Crop"]:
    df_yield[col] = df_yield[col].astype(str).str.strip()

# Exclude coconut and tree nut crops reported in nuts
df_yield = df_yield[~df_yield["Crop"].str.contains("Coconut", case=False, na=False)].copy()

# Compute Yield (Tonnes per Hectare)
df_yield["Yield"] = df_yield["Production"] / df_yield["Area"]

# Filter realistic agronomic range (0 < Yield <= 100 t/ha)
df_yield_clean = df_yield[(df_yield["Yield"] > 0) & (df_yield["Yield"] <= 100)].copy()

# Select top crops with sufficient samples for robust ML generalization
crop_counts = df_yield_clean["Crop"].value_counts()
top_crops = crop_counts.head(20).index.tolist()
df_yield_clean = df_yield_clean[df_yield_clean["Crop"].isin(top_crops)].copy()

print(f"Cleaned Yield Dataset: {len(df_yield_clean):,} records across {df_yield_clean['Crop'].nunique()} major crops.")
print("\nYield Statistics Summary (Tonnes / Hectare):")
print(df_yield_clean["Yield"].describe().round(3))


In [ ]:
# 3.2 Visualizing Crop Yield Distribution across Top Crops
plt.figure(figsize=(11, 4.5))
top_10 = df_yield_clean["Crop"].value_counts().head(10).index
sns.boxplot(data=df_yield_clean[df_yield_clean["Crop"].isin(top_10)], x="Crop", y="Yield", palette="Greens_r")
plt.xticks(rotation=30, ha="right")
plt.title("Yield Distribution (Tonnes/Ha) across Top 10 Indian Crops", fontsize=12, fontweight="bold")
plt.ylabel("Yield (Tonnes / Hectare)")
plt.tight_layout()
plt.show()


## 2. Train-Test Split (80/20)


In [ ]:
X = df_yield_clean[["Crop", "Season", "State_Name", "Area"]]
y = df_yield_clean["Yield"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train samples: {len(X_train):,} | Test samples: {len(X_test):,}")


## 3. Build Pipelines & Train Regressors: Linear Regression vs Random Forest


In [ ]:
# 5.1 Building Regression Pipelines & Training Models
categorical_features = ["Crop", "Season", "State_Name"]
numerical_features = ["Area"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

reg_models = {
    "Linear Regression": Pipeline([
        ("prep", preprocessor),
        ("reg", LinearRegression())
    ]),
    "Random Forest Regressor": Pipeline([
        ("prep", preprocessor),
        ("reg", RandomForestRegressor(n_estimators=50, max_depth=16, random_state=42, n_jobs=-1))
    ])
}

reg_metrics = {}

for name, pipe in reg_models.items():
    pipe.fit(X_yield_train, y_yield_train)
    y_pred = np.clip(pipe.predict(X_yield_test), 0, None)
    
    mae = mean_absolute_error(y_yield_test, y_pred)
    mse = mean_squared_error(y_yield_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_yield_test, y_pred)
    
    reg_metrics[name] = {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2_Score": r2,
        "y_pred_sample": y_pred[:500],
        "y_test_sample": y_yield_test.iloc[:500].values
    }

# Display Model Comparison Table
reg_summary_df = pd.DataFrame([
    {
        "Algorithm": k,
        "R² Score": f"{v['R2_Score']:.4f}",
        "MAE (t/ha)": f"{v['MAE']:.4f}",
        "RMSE (t/ha)": f"{v['RMSE']:.4f}",
        "MSE": f"{v['MSE']:.4f}"
    } for k, v in reg_metrics.items()
])
print("=== Regression Model Performance Comparison ===")
display(reg_summary_df)


In [ ]:
# 5.2 Visualizing R² Score Comparison & Actual vs Predicted Scatter
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# R2 Bar Chart
r2_vals = [reg_metrics[m]["R2_Score"] for m in reg_models.keys()]
sns.barplot(x=list(reg_models.keys()), y=r2_vals, palette=["#81C784", "#2E7D32"], ax=ax1)
ax1.set_ylim(0, 1.0)
ax1.set_title("R² Score Comparison (Variance Explained)", fontsize=11, fontweight="bold")
ax1.set_ylabel("R² Score")
for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.4f}", (p.get_x() + 0.25, p.get_height() / 2), color="white", fontweight="bold")

# Actual vs Predicted Scatter (Random Forest)
y_test_sub = reg_metrics["Random Forest Regressor"]["y_test_sample"]
y_pred_sub = reg_metrics["Random Forest Regressor"]["y_pred_sample"]
ax2.scatter(y_test_sub, y_pred_sub, alpha=0.5, color="#2E7D32", s=25)
max_val = max(np.max(y_test_sub), np.max(y_pred_sub))
ax2.plot([0, max_val], [0, max_val], "r--", label="Ideal 45° Line")
ax2.set_title("Actual vs Predicted Yield (Test Sample)", fontsize=11, fontweight="bold")
ax2.set_xlabel("Actual Yield (t/ha)")
ax2.set_ylabel("Predicted Yield (t/ha)")
ax2.legend()

plt.tight_layout()
plt.show()


## 4. Yield Prediction Function


In [ ]:
# 5.3 Crop Yield Prediction Function
best_yield_pipeline = reg_models["Random Forest Regressor"]

def predict_crop_yield(crop, season, state, area):
    """Predicts yield in Tonnes/Ha and calculates total harvest in Tonnes."""
    input_df = pd.DataFrame([[crop, season, state, float(area)]],
                            columns=["Crop", "Season", "State_Name", "Area"])
    
    pred_yield = float(best_yield_pipeline.predict(input_df)[0])
    pred_yield = max(0.01, round(pred_yield, 2))
    total_production = round(pred_yield * area, 2)
    
    print(f"🚜 CROP YIELD FORECAST for {crop} ({season} season in {state}):")
    print(f"   • Farm Size: {area:.1f} Hectares")
    print(f"   • Predicted Yield: {pred_yield:.2f} Tonnes / Hectare")
    print(f"   • Total Estimated Harvest: {total_production:,.2f} Tonnes")
    print(f"   • Model Used: Random Forest Regressor (R² = {reg_metrics['Random Forest Regressor']['R2_Score']:.4f})")
    return pred_yield, total_production

# Test Cases
print("--- TEST CASE 1: Punjab Wheat ---")
predict_crop_yield(crop="Wheat", season="Rabi", state="Punjab", area=10.0)

print("\n--- TEST CASE 2: Uttar Pradesh Sugarcane ---")
predict_crop_yield(crop="Sugarcane", season="Whole Year", state="Uttar Pradesh", area=5.0)
